# Notebook 01 - Embeddings: What They Are and Why They Matter

## What is an Embedding?

An embedding is a list of numbers that represents the **meaning** of a piece of text.

The idea is simple: if two pieces of text mean the same thing, their number lists should be similar.
If two pieces of text are completely unrelated, their number lists should be far apart.

This is not rule-based. A neural network (the embedding model) learns this from reading billions
of sentences during training. By the end of training, the model has figured out a way to compress
the meaning of any sentence into a fixed-length list of numbers.

## Why Not Just Use Keywords?

Keyword search looks for exact word matches.

These two sentences mean the same thing:
- "Our revenue grew by 20% this quarter."
- "The company's income increased significantly this period."

A keyword search for 'revenue' would find the first sentence and miss the second entirely.
An embedding model would give both sentences very similar vectors, so both would appear in a
semantic search result.

That is the core advantage of embeddings over keyword search.

## The Model We Use

We use `sentence-transformers` with the model `all-MiniLM-L6-v2`.

- Runs entirely on your machine — no API key needed
- Downloads once (~90 MB) and is cached locally after that
- Produces embeddings of dimension 384
- Fast enough for interactive use on CPU

In [ ]:
# Install dependencies (run once)
# !pip install sentence-transformers numpy

In [1]:
import sys
sys.path.append('..')

from utils.embedder import embed, embed_batch, cosine_similarity
import numpy as np

d:\Naveena Natarajan\Tutor\Abhishek-Bangalore-Python\Abhishek_Implementation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
!pip install hf_xet

## Part 1 - Your First Embedding

Let's embed one sentence and look at what comes out.

In [4]:
sentence = "Our revenue grew by 20% this quarter."

vector = embed(sentence)

print("Input     :", sentence)
print("Type      :", type(vector))
print("Shape     :", vector.shape)
print("First 10  :", vector[:10].round(4))
print("Min / Max :", round(vector.min(), 4), "/", round(vector.max(), 4))

Input     : Our revenue grew by 20% this quarter.
Type      : <class 'numpy.ndarray'>
Shape     : (384,)
First 10  : [ 0.0609 -0.0232 -0.0142  0.0073  0.0308 -0.0026 -0.0103  0.0539 -0.0008
  0.0259]
Min / Max : -0.1515 / 0.1259




## Part 2 - Comparing Two Sentences

Cosine similarity measures how similar two vectors are.

- Value of 1.0 means identical direction (same meaning)
- Value of 0.0 means perpendicular (unrelated)
- Value close to -1.0 means opposite meaning

For normalized vectors (which `embed()` produces), cosine similarity equals the dot product.

In [5]:
s1 = "Our revenue grew by 20% this quarter."
s2 = "The company's income increased significantly this period."
s3 = "Preheat the oven to 180 degrees before baking."

v1 = embed(s1)
v2 = embed(s2)
v3 = embed(s3)

print("Similarity: S1 vs S2 (revenue vs income):", round(cosine_similarity(v1, v2), 4))
print("Similarity: S1 vs S3 (revenue vs cooking):", round(cosine_similarity(v1, v3), 4))
print("Similarity: S2 vs S3 (income vs cooking) :", round(cosine_similarity(v2, v3), 4))

Similarity: S1 vs S2 (revenue vs income): 0.5108
Similarity: S1 vs S3 (revenue vs cooking): -0.0059
Similarity: S2 vs S3 (income vs cooking) : 0.044


The revenue and income sentences score high because the model understands they are synonymous in a business context.

The cooking sentence scores low against both - it belongs to a completely different semantic space.

## Part 3 - The Revenue / Income / Earnings Experiment

Embed 5 sentences and build a full similarity matrix.
This is the experiment that demonstrates why embeddings matter for retrieval.

In [6]:
sentences = [
    "Our revenue grew by 20% this quarter.",             # S1
    "The company income increased significantly.",        # S2
    "Quarterly earnings show a strong upward trend.",    # S3
    "Handle price objections by focusing on ROI.",       # S4
    "Preheat the oven to 180 degrees for baking.",       # S5
]

labels = ["S1-Revenue", "S2-Income", "S3-Earnings", "S4-Objection", "S5-Cooking"]

# Batch embed — faster than one by one
vectors = embed_batch(sentences)

Batches: 100%|██████████| 1/1 [00:00<00:00, 40.47it/s]


In [7]:
# Build and print the similarity matrix
n = len(sentences)

print(f"{'':>14}", end="")
for label in labels:
    print(f"{label:>14}", end="")
print()
print("-" * (14 * (n + 1)))

for i in range(n):
    print(f"{labels[i]:>14}", end="")
    for j in range(n):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"{sim:>14.4f}", end="")
    print()

                  S1-Revenue     S2-Income   S3-Earnings  S4-Objection    S5-Cooking
------------------------------------------------------------------------------------
    S1-Revenue        1.0000        0.5195        0.4733        0.2898        0.0021
     S2-Income        0.5195        1.0000        0.4918        0.3136        0.0334
   S3-Earnings        0.4733        0.4918        1.0000        0.2510        0.0456
  S4-Objection        0.2898        0.3136        0.2510        1.0000        0.0865
    S5-Cooking        0.0021        0.0334        0.0456        0.0865        1.0000




## What to Notice in the Matrix

- S1, S2, S3 (revenue, income, earnings) should all be similar to each other.
  They describe the same business concept using different words.

- S4 (objection handling) should be moderately related to the financial sentences
  because they are both in a business context.

- S5 (cooking) should be near zero against everything else.
  The model has correctly isolated it as belonging to a different domain.

This is what makes semantic search powerful. The retrieval system does not match words.
It matches meaning.


## Part 4 - What Do the Numbers Actually Represent?

Each position in the 384-number vector captures some aspect of meaning.
The model learned these dimensions automatically during training - they do not have
human-readable names like 'formality' or 'sentiment'.

But we can observe the effect: sentences with similar meaning produce vectors
that point in nearly the same direction in 384-dimensional space.

The diagram below shows a simplified 2D version of what is happening in 384 dimensions.



## Summary

- An embedding is a fixed-length list of numbers that encodes the meaning of text.
- The embedding model (sentence-transformers) was trained to place similar meanings close together.
- Cosine similarity measures how close two embeddings are. Range: -1 to 1. Higher = more similar.
- This is the foundation of semantic search and, by extension, the RAG pipeline.

Next: Notebook 02 shows how to use FAISS to search through thousands of embeddings efficiently.